# Qui exploite les mines guinéennes, et que touche l'État ? (2019-2024)

**Question :** quelles entreprises produisent et exportent la bauxite et l'or de la Guinée, la production est-elle concentrée, et combien chaque entreprise verse-t-elle à l'État ?

**Sources** (données par entreprise, recopiées depuis les PDF officiels ; source et page indiquées pour chaque ligne) :
- Ministère des Mines et de la Géologie (MMG), Annuaire statistique des mines et carrières 2023 : tableaux 11, 13, 14 et 25 (2019-2023).
- MMG, Bulletin annuel 2024 : tableaux I-1, I-2, IV-1, VI-1 et VI-2 (2024).
- INS, Note d'analyse du commerce extérieur 2024 : valeur des exportations (étape 1).

Fichiers : `data/bauxite_par_entreprise.csv`, `data/or_par_entreprise.csv`, `data/taxes_bauxite_par_entreprise.csv`.

In [ ]:
import sys
sys.path.append("..")

import duckdb
from src.graphiques import graphique_parts_entreprises, graphique_evolution_entreprises, graphique_taxe_par_tonne

BAUXITE = "'../data/bauxite_par_entreprise.csv'"
OR = "'../data/or_par_entreprise.csv'"
TAXES = "'../data/taxes_bauxite_par_entreprise.csv'"
NAT = "'../data/sources_nationales.csv'"

TAUX_USD_GNF_2024 = 8601.6963   # BCRG, 31/12/2024 (cité dans le Bulletin annuel 2024 du MMG)

## 0. Contrôle qualité : les totaux par entreprise correspondent-ils aux totaux officiels ?

In [ ]:
duckdb.sql(f"""
    SELECT b.annee,
           SUM(b.production_t)::BIGINT AS somme_entreprises,
           MAX(n.valeur)::BIGINT       AS total_publie,
           SUM(b.production_t)::BIGINT - MAX(n.valeur)::BIGINT AS ecart
    FROM {BAUXITE} b
    JOIN {NAT} n ON n.annee = b.annee AND n.indicateur = 'bauxite_production'
    GROUP BY b.annee
    ORDER BY b.annee
""").show()

**Constat :** pour 2022, la somme des entreprises donne 103,5 Mt alors que l'Annuaire 2023 publie un total de 94,5 Mt. C'est le total de l'Annuaire qui est faux : le Bulletin 2024 du même ministère publie bien 103,5 Mt pour 2022. Cela explique l'incohérence repérée à l'étape 1. Pour 2023, l'écart est faible (0,7 Mt, soit 0,6 %).

## 1. Une production très concentrée

In [ ]:
parts = duckdb.sql(f"""
    SELECT entreprise,
           exportation_t / 1e6 AS millions_tonnes,
           100 * exportation_t / SUM(exportation_t) OVER () AS part_pct,
           100 * SUM(exportation_t) OVER (ORDER BY exportation_t DESC)
               / SUM(exportation_t) OVER () AS part_cumulee_pct
    FROM {BAUXITE}
    WHERE annee = 2024
    ORDER BY exportation_t DESC
""").df()
print(parts.round(1))
graphique_parts_entreprises(parts, 2024)

**Conclusion :** en 2024, SMB exporte à elle seule 38 % de la bauxite guinéenne (55 Mt). Les trois premières entreprises (SMB, AGB2A SDM, CBG) font 62 % des exportations, et les six premières 86 %. Le secteur dépend donc de quelques acteurs : l'arrêt d'une seule grande mine pèserait lourd sur les exportations du pays.

## 2. Qui a porté le boom 2019-2024 ?

In [ ]:
evolution = duckdb.sql(f"""
    SELECT annee, entreprise, exportation_t / 1e6 AS millions_tonnes
    FROM {BAUXITE}
    WHERE entreprise IN (
        SELECT entreprise FROM {BAUXITE} WHERE annee = 2024
        ORDER BY exportation_t DESC LIMIT 6
    )
""").df()
tableau = evolution.pivot(index="annee", columns="entreprise", values="millions_tonnes")
print(tableau.round(1))
graphique_evolution_entreprises(tableau, a_souligner=["SMB", "AGB2A SDM"])

In [ ]:
# Contribution de chaque entreprise à la hausse 2019 -> 2024
duckdb.sql(f"""
    WITH v AS (
        SELECT entreprise,
               COALESCE(SUM(exportation_t) FILTER (WHERE annee = 2019), 0) AS t2019,
               COALESCE(SUM(exportation_t) FILTER (WHERE annee = 2024), 0) AS t2024
        FROM {BAUXITE}
        GROUP BY entreprise
    )
    SELECT entreprise,
           ROUND((t2024 - t2019) / 1e6, 1) AS hausse_mt,
           ROUND(100 * (t2024 - t2019) / SUM(t2024 - t2019) OVER (), 1) AS part_de_la_hausse_pct
    FROM v
    ORDER BY hausse_mt DESC
    LIMIT 6
""").show()

**Conclusion :** entre 2019 et 2024, les exportations de bauxite ont augmenté de près de 80 Mt. SMB (+20 Mt) et AGB2A SDM (+19 Mt, partie de zéro en 2020) en expliquent environ la moitié. Viennent ensuite CHALCO et GAC, qui démarrent ou décollent en 2020. CBG, l'exploitant historique, reste stable autour de 16 Mt.

## 3. Que touche l'État par tonne exportée ?

In [ ]:
taxes = duckdb.sql(f"""
    SELECT t.entreprise, t.taxe_miniere_usd, b.exportation_t,
           t.taxe_miniere_usd / b.exportation_t AS usd_par_tonne
    FROM {TAXES} t
    JOIN {BAUXITE} b USING (annee, entreprise)
    WHERE t.annee = 2024 AND t.taxe_miniere_usd IS NOT NULL
""").df()
print(taxes.round(2))
graphique_taxe_par_tonne(taxes, 2024)

In [ ]:
duckdb.sql(f"""
    WITH recettes AS (
        SELECT SUM(taxe_miniere_usd) + SUM(droits_douane_usd) + SUM(contribution_communautaire_usd) AS total_usd
        FROM {TAXES}
        WHERE annee = 2024
    ),
    exportations AS (
        SELECT valeur * 1e9 / {TAUX_USD_GNF_2024} AS valeur_usd
        FROM {NAT}
        WHERE indicateur = 'export_bauxite' AND annee = 2024
    )
    SELECT ROUND(r.total_usd / 1e6)       AS recettes_bauxite_millions_usd,
           ROUND(e.valeur_usd / 1e9, 2)   AS exportations_bauxite_milliards_usd,
           ROUND(100 * r.total_usd / e.valeur_usd, 1) AS part_pour_l_etat_pct
    FROM recettes r, exportations e
""").show()

**Conclusion :** en 2024, la taxe minière payée par tonne exportée va de 1,5 $ (BAM) à 3,8 $ (AGB2A SDM), avec une moyenne d'environ 3,1 $ pour les entreprises qui la paient. Au total, les sociétés bauxitiques ont versé environ 428 millions de dollars (taxe minière, douane, contribution communautaire), soit environ 9 % de la valeur des exportations de bauxite (4,7 milliards de dollars selon l'INS).

**Limites à signaler :**
- Ce calcul ne compte que les paiements publiés par le ministère. Il manque l'impôt sur les sociétés et les dividendes de l'État dans les sociétés où il est actionnaire, publiés dans les rapports ITIE.
- CBG ne paie pas de taxe minière : elle est régie par une convention spéciale, et l'État guinéen en est actionnaire.
- Les écarts de taxe par tonne entre entreprises peuvent venir de prix de vente, de qualités de bauxite ou de conventions différentes. Les données ne permettent pas de trancher.

## 4. L'or : quelques entreprises industrielles

In [ ]:
duckdb.sql(f"""
    SELECT entreprise,
           SUM(production_kg) FILTER (WHERE annee = 2019) AS kg_2019,
           SUM(production_kg) FILTER (WHERE annee = 2023) AS kg_2023,
           SUM(production_kg) FILTER (WHERE annee = 2024) AS kg_2024,
           ROUND(100 * SUM(production_kg) FILTER (WHERE annee = 2024)
                 / SUM(SUM(production_kg) FILTER (WHERE annee = 2024)) OVER (), 1) AS part_2024_pct
    FROM {OR}
    GROUP BY entreprise
    ORDER BY kg_2024 DESC NULLS LAST
""").show()

**Conclusion :** la Société AngloGold Ashanti de Guinée (SAG) produit à elle seule près de la moitié de l'or industriel du pays (45 % en 2024), suivie de SMD et SMM. L'or industriel ne représente qu'une partie de l'or exporté : l'orpaillage artisanal en exporte davantage (environ 52 t en 2024 selon le même bulletin), ce qui rend l'or bien plus difficile à suivre et à taxer que la bauxite.

## Ce qu'il faut retenir

1. **Un secteur concentré :** SMB exporte 38 % de la bauxite guinéenne, et 3 entreprises en exportent 62 % (2024).
2. **Le boom vient surtout de deux acteurs :** SMB et AGB2A SDM expliquent environ la moitié des 80 Mt supplémentaires exportées entre 2019 et 2024.
3. **L'État touche environ 9 % de la valeur exportée** en taxes minières, droits de douane et contributions communautaires, soit environ 3 $ de taxe minière par tonne pour les entreprises qui la paient.
4. **La qualité des données compte :** le total 2022 publié par l'Annuaire était faux de 9 Mt ; la somme par entreprise et le Bulletin 2024 l'ont révélé.

**Prochaine étape :** à qui la Guinée vend sa bauxite (données miroir UN Comtrade), et la place de la Chine.